# Урок 31. Итоговый проект: данные

**Дата:** 22.04.2027 · **60 минут** · 8–9 класс  
**Модуль 6.** Итоговый проект (уроки 30–36)  
**Опирается на:** урок 30 (паспорт проекта), уроки 10–14 (pandas, очистка, разведка, графики), урок 17 (утечки), урок 24 (приватность), уроки 26–27 (картинки и тексты).

Данные — фундамент проекта: никакая модель не исправит плохие данные. Сегодня доводим данные до состояния «готово к модели» и заполняем **журнал данных** — его обязательно спросят на защите.

## Цели урока

1. Собрать данные проекта и проверить, что их достаточно.
2. Провести паспорт, очистку и разведку своих данных.
3. Проверить данные на утечки и этические риски.
4. Заполнить журнал данных и карточку датасета.

## План на 60 минут

| Минуты | Что делаем |
|-------:|------------|
| 0–5 | Чек-лист урока, вопросы по сбору данных |
| 5–45 | Работа над данными по своему типу проекта |
| 45–55 | Журнал данных, карточка датасета |
| 55–60 | Короткий отчёт: где я сейчас |

## 1. Сколько данных нужно

| Тип проекта | Минимум | Хорошо |
|-------------|---------|--------|
| Таблица (классификация / регрессия) | 80 строк | 150+ строк |
| Картинки | 30 фото на класс | 50+ на класс, разные условия |
| Текст / бот | 10 фраз на намерение | 15–20, от разных людей |

**Если данных мало:** сократи число классов, объедини похожие, попроси ещё нескольких одноклассников заполнить опрос. Нельзя: придумывать данные и выдавать их за настоящие. Если используешь сгенерированные или учебные данные — **честно напиши** об этом в журнале.

## 2. Сбор данных: правила

- **Опрос** — анонимный (без ФИО), с объяснением, зачем он нужен. Вопросы с вариантами ответов собирать проще, чем свободный текст.
- **Фото** — только с разрешения людей, лучше без лиц; разные фоны, свет, ракурсы.
- **Фразы для бота** — попроси 3–5 человек написать, как бы они спросили; не пиши все фразы сам.
- **Открытые наборы данных** — укажи источник и проверь, что данные можно использовать для учёбы.

## 3. Табличный проект: шаблон работы с данными

Ячейка ниже создаёт **пример** файла опроса, чтобы шаблон работал. Замени его на свой файл: загрузи CSV через панель файлов Colab (урок 8) и укажи его имя в `read_csv`. Если делаешь опрос в Google Форме — ответы можно скачать из Google Таблиц как CSV.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ПРИМЕР ДАННЫХ — замени на свой файл
with open("project_example.csv", "w", encoding="utf-8") as f:
    f.write("дней_до_дедлайна,сделано_процентов,уроков_в_день,кружков,успел\n")
    f.write("7,20,6,2,да\n")
    f.write("3,10,7,3,нет\n")
    f.write("10,50,6,1,да\n")
    f.write("2,80,5,2,да\n")
    f.write("5,,6,4,нет\n")
    f.write("7,20,6,2,да\n")
    f.write("1,30,7,3,нет\n")
    f.write("6,60,6,1,Да\n")
    f.write("4,15,8,2,нет\n")
    f.write("9,40,5,0,да\n")

df = pd.read_csv("project_example.csv")   # ← здесь твой файл
df.head()

### Шаг 1. Паспорт

In [ ]:
print("Размер:", df.shape)
df.info()
print(df.isna().sum())
print("Дубликатов:", df.duplicated().sum())
df.describe()

### Шаг 2. Очистка

Повтори приёмы урока 11 и **запиши каждое решение в журнал данных** (раздел 6).

In [ ]:
clean = df.copy()

# Пример: разное написание ответов
clean["успел"] = clean["успел"].str.strip().str.lower()

# Пример: дубликаты
clean = clean.drop_duplicates()

# Пример: пропуски — заполним медианой и запомним, где было пусто
clean["сделано_пропуск"] = clean["сделано_процентов"].isna()
clean["сделано_процентов"] = clean["сделано_процентов"].fillna(clean["сделано_процентов"].median())

# Цель — в True / False
clean["успел"] = clean["успел"] == "да"

clean = clean.reset_index(drop=True)
print(clean.shape)
clean

### Шаг 3. Разведка

Главный показатель (цель): какое соотношение классов? Есть ли связь признаков с целью? Не забудь про размер групп (урок 12).

In [ ]:
print(clean["успел"].value_counts())
print(clean.groupby("успел")[["дней_до_дедлайна", "сделано_процентов", "уроков_в_день", "кружков"]].mean().round(1))
print(clean[["дней_до_дедлайна", "сделано_процентов", "уроков_в_день", "кружков"]].corr().round(2))

### Шаг 4. Хотя бы один график с выводом

In [ ]:
plt.figure(figsize=(6, 4))
done = clean[clean["успел"]]
late = clean[~clean["успел"]]
plt.scatter(done["дней_до_дедлайна"], done["сделано_процентов"], color="seagreen", label="успел")
plt.scatter(late["дней_до_дедлайна"], late["сделано_процентов"], color="crimson", label="не успел")
plt.title("Дни до дедлайна, готовность и результат")
plt.xlabel("Дней до дедлайна")
plt.ylabel("Сделано, %")
plt.legend()
plt.show()
print("Вывод: …")

### Шаг 5. Сохрани чистые данные

In [ ]:
clean.to_csv("project_clean.csv", index=False)
print("Сохранено: project_clean.csv")

## 4. Текстовый проект: проверка данных

Для бота данные — это фразы и намерения. Проверь: сколько фраз на каждое намерение, нет ли повторов, нет ли намерений-«сирот» с 2–3 фразами.

In [ ]:
phrases = [
    ("когда тренировка", "расписание"), ("во сколько секция", "расписание"),
    ("где спортзал", "место"), ("как пройти в зал", "место"),
    ("сколько стоит абонемент", "цена"), ("когда тренировка", "расписание"),
]   # ← здесь твои фразы

texts = []
intents = []
for text, intent in phrases:
    texts.append(text)
    intents.append(intent)

table = pd.DataFrame({"фраза": texts, "намерение": intents})
print(table["намерение"].value_counts())
print("Повторов фраз:", table.duplicated().sum())

**Правила разметки** (запиши в журнал): как ты решаешь спорные случаи? Например: «Фраза одновременно про время и место — отношу к намерению, которое стоит первым в вопросе». Одинаковые правила делают разметку честной.

## 5. Проект с картинками: проверка данных

- [ ] в каждом классе примерно одинаковое число фото;
- [ ] фото сделаны в **разных** условиях: свет, фон, расстояние, ракурс;
- [ ] фон не «подсказывает» класс (все бутылки не сняты на одном столе, а бумага — на другом);
- [ ] отложены **отдельные** фото для проверки, сделанные в другой день или другом месте;
- [ ] нет лиц и личных вещей с данными.

## 6. Журнал данных (обязателен на защите)

| Пункт | Мой ответ |
|-------|-----------|
| Источник данных | |
| Как собирал, когда, сколько человек участвовало | |
| Согласие и обезличивание | |
| Было строк / фото / фраз | |
| Стало после очистки | |
| Проблемы и решения (пропуски, дубликаты, опечатки, выбросы) | |
| Правила разметки (для текста и картинок) | |
| Утечки: какие столбцы исключил и почему | |
| Соотношение классов / диапазон цели | |
| Главные наблюдения разведки (с числами) | |
| Ограничения данных (кого / чего в данных нет) | |

## 7. Проверка на утечку и этический фильтр

Ответь честно на каждый вопрос:

- [ ] Каждый признак известен **в момент предсказания**? (урок 17)
- [ ] Нет ли признака, который почти совпадает с целью?
- [ ] Нет ли в данных ФИО, телефонов, адресов, фото лиц?
- [ ] Все участники согласились на использование данных?
- [ ] Нет ли группы людей, которой в данных почти нет, а модель будет применяться и к ней? (урок 24)

## Работа с ИИ-помощником

- «Вот названия столбцов моего опроса: … Есть ли среди них утечка для цели …?»
- «Предложи 3 вопроса для анонимного опроса, чтобы собрать признак …»
- Не отправляй помощнику сами ответы опроса — только названия столбцов и обезличенные примеры.

Запиши вопросы в журнал использования ИИ.

## Домашнее задание

1. Доделай сбор и очистку данных, сохрани `project_clean.csv` (или папку фото / список фраз).
2. Заполни журнал данных полностью.
3. Подготовь вопрос преподавателю, если что-то не получается — на следующем уроке строим модель.

## Самопроверка

1. Сколько у тебя примеров? Достаточно ли их по таблице из раздела 1?
2. Какие проблемы были в данных и как ты их решил?
3. Какие признаки ты исключил и почему?
4. Каких людей или ситуаций нет в твоих данных?

**Следующий урок (29.04.2027):** проект — модель и метрики.